In [ ]:
import os
import time  # Import the time module
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.nn as nn 
import torch.optim as optim 
from sklearn.model_selection import train_test_split

In [ ]:
def load_matrix_dataset(export_dir, delimiter=',', batch_size=16):
    """
    Loads matrix data from CSVs, automatically infers classes, and 
    returns a PyTorch DataLoader and the generated label mapping.
    """
    if not os.path.exists(export_dir):
        raise ValueError(f"Directory not found: {export_dir}")

    X_list = []
    y_list = []
    label_map = {}
    current_label_idx = 0

    for filename in os.listdir(export_dir):
        if filename.endswith(".csv"):
            filepath = os.path.join(export_dir, filename)
            
            # 1. Flexible Matrix Loading
            # np.loadtxt automatically adapts to whatever N x M shape the CSV is
            matrix = np.loadtxt(filepath, delimiter=delimiter)
            X_list.append(matrix)
            
            # 2. Dynamic Label Extraction
            # We split by the *last* underscore to support complex class names.
            # Example: "Japan_female_1.csv" -> Class: "Japan_female", ID: "1"
            base_name = os.path.splitext(filename)[0]
            
            if '_' in base_name:
                label_name = base_name.rsplit('_', 1)[0] 
            else:
                # Fallback if there is no underscore identifier
                label_name = base_name 
            
            # 3. Auto-build the Label Map
            if label_name not in label_map:
                label_map[label_name] = current_label_idx
                current_label_idx += 1
                
            y_list.append(label_map[label_name])

    if not X_list:
        raise ValueError(f"No CSV files found in {export_dir}")

    # 4. Consistency Check
    # PyTorch requires all matrices in a single tensor to have identical dimensions.
    shapes = [x.shape for x in X_list]
    unique_shapes = set(shapes)
    if len(unique_shapes) > 1:
        raise ValueError(f"Inconsistent matrix dimensions found: {unique_shapes}. "
                         f"All matrices in the dataset must be the exact same size.")

    # 5. Convert to PyTorch Tensors
    X_tensor = torch.tensor(np.array(X_list), dtype=torch.float32).unsqueeze(1)
    y_tensor = torch.tensor(np.array(y_list), dtype=torch.long)

    # 6. Create DataLoader
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    print(f"Successfully loaded {len(X_tensor)} matrices.")
    print(f"Matrix Dimension: {X_tensor.shape[2:]}")
    print(f"Classes found ({len(label_map)}): {label_map}")

    return dataloader, X_tensor, y_tensor, label_map

In [ ]:
class GeneralizedMatrixCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(GeneralizedMatrixCNN, self).__init__()
        
        # 1. Feature Extraction (No hardcoded pooling sizes)
        self.features = nn.Sequential(
            # We use padding="same" so the convolution doesn't shrink the edges
            nn.Conv2d(1, 16, kernel_size=3, padding="same"),
            nn.ReLU(),
            # Optional: A standard gentle pool. If your matrices might be extremely 
            # tiny (like 2x2), you can comment this line out.
            nn.MaxPool2d(kernel_size=2, stride=2), 
            
            nn.Conv2d(16, 32, kernel_size=3, padding="same"),
            nn.ReLU(),
        )
        
        # 2. The Magic Layer: Adaptive Pooling
        # This takes ANY (Batch, 32, H, W) and turns it into (Batch, 32, 1, 1)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 3. The Classifier
        # The input is now ALWAYS 32, regardless of N or M
        self.classifier = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        
        # Flatten the (Batch, 32, 1, 1) tensor into (Batch, 32)
        x = x.view(x.size(0), -1) 
        
        x = self.classifier(x)
        return x

Random Split

In [ ]:
def run_random_split(X_tensor, y_tensor, epochs=150, patience=15):
    print("\n" + "="*40)
    print("STARTING METHOD 1: 75/25 RANDOM SPLIT")
    print("="*40)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    total_size = len(dataset)
    train_size = int(0.75 * total_size)
    test_size = total_size - train_size

    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


    num_classes = len(torch.unique(y_tensor))
    print(f"Detected {num_classes} unique classes. Initializing model...")

    model = GeneralizedMatrixCNN(num_classes=num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_test_loss = float('inf')
    epochs_no_improve = 0
    best_model_path = 'temp_best_split_model.pth'

    for epoch in range(epochs):
        # Train
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        running_test_loss = 0.0
        with torch.no_grad():
            for test_X, test_y in test_loader:
                t_loss = criterion(model(test_X), test_y)
                running_test_loss += t_loss.item() * test_X.size(0)
                
        avg_test_loss = running_test_loss / test_size
        
        # Early Stopping
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

    # Final Test Accuracy on Best Weights
    model.load_state_dict(torch.load(best_model_path))
    model.eval() 
    correct = 0
    with torch.no_grad():
        for test_X, test_y in test_loader:
            _, predicted = torch.max(model(test_X).data, 1)
            correct += (predicted == test_y).sum().item()

    accuracy = (correct / test_size) * 100
    print(f"Random Split Result: {accuracy:.2f}% ({correct}/{test_size} correct)")
    return accuracy

In [ ]:
def run_stratified_split(X_tensor, y_tensor, epochs=150, patience=15):
    print("\n" + "="*40)
    print("STARTING METHOD 1: 75/25 STRATIFIED SPLIT")
    print("="*40)
    
    # 1. Stratified Split using scikit-learn
    # By passing stratify=y_tensor, it guarantees balanced class ratios
    X_train, X_test, y_train, y_test = train_test_split(
        X_tensor, 
        y_tensor, 
        test_size=0.25, 
        stratify=y_tensor, 
        random_state=42 # Set a seed for reproducible splits
    )

    # 2. Re-wrap into PyTorch Datasets
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    test_size = len(test_dataset)
    print(f"Training on {len(train_dataset)} samples | Testing on {test_size} samples")

    # 3. Initialize Model
    num_classes = len(torch.unique(y_tensor))
    print(f"Detected {num_classes} unique classes. Initializing model...")

    model = GeneralizedMatrixCNN(num_classes=num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_test_loss = float('inf')
    epochs_no_improve = 0
    best_model_path = 'temp_best_stratified_model.pth'

    # 4. Training Loop with Early Stopping
    for epoch in range(epochs):
        # Train
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        running_test_loss = 0.0
        with torch.no_grad():
            for test_X, test_y in test_loader:
                t_loss = criterion(model(test_X), test_y)
                running_test_loss += t_loss.item() * test_X.size(0)
                
        avg_test_loss = running_test_loss / test_size
        
        # Early Stopping Logic
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

    # 5. Final Evaluation on Best Weights
    model.load_state_dict(torch.load(best_model_path))
    model.eval() 
    correct = 0
    
    with torch.no_grad():
        for test_X, test_y in test_loader:
            _, predicted = torch.max(model(test_X).data, 1)
            correct += (predicted == test_y).sum().item()

    accuracy = (correct / test_size) * 100
    print(f"Stratified Split Result: {accuracy:.2f}% ({correct}/{test_size} correct)")
    return accuracy

LOOCV

In [ ]:
def run_loocv(X_tensor, y_tensor, epochs_per_fold=30):
    print("\n" + "="*40)
    print("STARTING METHOD 2: LOOCV")
    print("="*40)
    
    N = len(X_tensor)
    correct_predictions = 0

    for i in range(N):
        # Isolate 1 test sample, keep N-1 for training
        test_X = X_tensor[i:i+1]
        test_y = y_tensor[i:i+1]
        
        train_X = torch.cat((X_tensor[:i], X_tensor[i+1:]), dim=0)
        train_y = torch.cat((y_tensor[:i], y_tensor[i+1:]), dim=0)
        
        # CRITICAL: Fresh model for every fold
        num_classes = len(torch.unique(y_tensor))
        print(f"Detected {num_classes} unique classes. Initializing model...")

        model = GeneralizedMatrixCNN(num_classes=num_classes)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        # Train (Full-batch gradient descent is faster for small N)
        model.train()
        for epoch in range(epochs_per_fold):
            optimizer.zero_grad()
            loss = criterion(model(train_X), train_y)
            loss.backward()
            optimizer.step()
            
        # Test on the held-out sample
        model.eval()
        with torch.no_grad():
            _, predicted = torch.max(model(test_X).data, 1)
            if (predicted == test_y).item():
                correct_predictions += 1
                
        # Progress update
        if (i + 1) % 10 == 0 or i == 0:
            print(f"LOOCV Progress: {i+1}/{N} folds complete...")

    accuracy = (correct_predictions / N) * 100
    print(f"LOOCV Final Result: {accuracy:.2f}% ({correct_predictions}/{N} correct)")
    return accuracy

In [ ]:
# Define your list of target directories
export_dirs = [
    "energy/matrix_export",
    "HMD/matrix_export",
    "gait/matrix_export",
    "sim_data/matrix_export1",
    "sim_data/matrix_export2",
    "sim_data/matrix_export3",
    "sim_data/matrix_export4"
]


results = {}

for path in export_dirs:
    category = path.split('/')[0]
    print(f"--- Processing Category: {category} ---")
    
    # Start the timer for this path
    start_time = time.perf_counter()

    # Load dataset
    dataloader, X_tensor, y_tensor, label_map = load_matrix_dataset(path)
    
    # Logic Branching: Use Stratified Split specifically for HMD
    if category == "HMD":
        print("Executing Stratified Split for HMD...")
        split_acc = run_stratified_split(X_tensor, y_tensor, epochs=150, patience=15)
    else:
        print(f"Executing Random Split for {category}...")
        split_acc = run_random_split(X_tensor, y_tensor, epochs=150, patience=15)
    
    # Run Leave-One-Out Cross-Validation (Method 2)
    #loocv_acc=1
    loocv_acc = run_loocv(X_tensor, y_tensor, epochs_per_fold=40)
    
    # Stop the timer and calculate the duration
    end_time = time.perf_counter()
    run_time = end_time - start_time
    
    print(f"Finished {path} in {run_time:.2f} seconds.")

    # Store results
    results[category] = {
        "split_accuracy": split_acc,
        "loocv_accuracy": loocv_acc
    }

print("\nAll tasks completed.")